# Restaurant Booking Agent on Amazon Bedrock AgentCore

Build and deploy a restaurant assistant on **Amazon Bedrock AgentCore**.

### Use case

A restaurant agent that can:

1. Answer questions about the menu, from an **existing Bedrock Knowledge Base**
2. **Create** a table booking
3. **Get** the details of a booking
4. **Delete** a booking

Bookings are stored in a DynamoDB table called `restaurant_bookings`.

### Architecture

![AgentCore architecture](./images/agentcore-architecture.png)

A customer talks to one agent running on AgentCore Runtime. The agent branches to booking
storage on one side and the menu knowledge base on the other. Its tools run inside the
runtime container and talk to DynamoDB directly. Memory and Observability come from the
runtime itself.

There are two more diagrams later in this notebook, because a system like this really has
three views and mixing them into one picture makes all three unreadable:

| Diagram | Where | What it shows |
| --- | --- | --- |
| **Agent architecture** (above) | here | What the agent is made of |
| **Build and deploy** | Step 6 | How your code becomes a running agent, once |
| **End to end** | Step 10 | The full request path from a customer's browser |

### What you will use

| AgentCore capability | What it gives you here |
| --- | --- |
| **Runtime** | Serverless hosting for the agent, with session isolation |
| **Memory** | Multi-turn conversation - the agent follows up on what was said before |
| **Observability** | Traces of every model call and tool call, in CloudWatch |
| **Strands Agents SDK** | The agent loop and the `@tool` decorator |
| **Bedrock Knowledge Base** | Menu retrieval, reached through an ordinary tool |

The agent itself is a single file, `restaurant_agent.py`, that you write from this
notebook. Everything else here is deployment.

## Prerequisites

**Permissions.** The identity running this notebook needs to be able to create IAM roles,
DynamoDB tables, ECR repositories, CodeBuild projects, and AgentCore resources. On SageMaker
Studio the simplest route is to attach:

- `IAMFullAccess`
- `AmazonDynamoDBFullAccess`
- `AmazonBedrockFullAccess`
- `BedrockAgentCoreFullAccess`
- `AmazonEC2ContainerRegistryFullAccess`
- `AWSCodeBuildDeveloperAccess`

**Model access.** Enable the model you intend to use in the Amazon Bedrock console. This
notebook defaults to `us.amazon.nova-2-lite-v1:0`.

**Knowledge Base.** You need an existing Bedrock Knowledge Base. You will paste its ID in
the configuration cell below.

**No Docker needed.** `runtime.launch()` builds the ARM64 container in AWS CodeBuild, so this
works from Windows, macOS, Linux or SageMaker Studio without a local container runtime.

## Step 0 - Install dependencies

`bedrock-agentcore-starter-toolkit` is what gives us the notebook-friendly
`Runtime` and `Observability` classes. Restart the kernel after this cell if pip
upgrades a package that is already imported.

In [ ]:
%pip install -q -U -r notebook-requirements.txt

## Step 1 - Configuration

This is the only cell you need to edit.

> **Set `KNOWLEDGE_BASE_ID` to your own Knowledge Base ID.** You can find it in the
> Amazon Bedrock console under *Knowledge Bases*, or list them with the cell below.
> It looks like `AHA1FA3Z46`. Nothing else in the notebook needs changing.

In [ ]:
import boto3

session = boto3.session.Session()
REGION = session.region_name or "us-east-1"
ACCOUNT_ID = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]

# ---- edit me -------------------------------------------------------------
KNOWLEDGE_BASE_ID = "AHA1FA3Z46"          # <-- your existing Knowledge Base ID
# --------------------------------------------------------------------------

AGENT_NAME = "booking_agent"               # letters, digits and _ only; max 48 chars
MODEL_ID = "us.amazon.nova-2-lite-v1:0"
TABLE_NAME = "restaurant_bookings"
ENTRYPOINT = "restaurant_agent.py"

print(f"Region:         {REGION}")
print(f"Account:        {ACCOUNT_ID}")
print(f"Agent name:     {AGENT_NAME}")
print(f"Model:          {MODEL_ID}")
print(f"Knowledge Base: {KNOWLEDGE_BASE_ID}")

Not sure which Knowledge Base to use? Run this to list the ones in your account.

In [ ]:
bedrock_agent = boto3.client("bedrock-agent", region_name=REGION)

for kb in bedrock_agent.list_knowledge_bases().get("knowledgeBaseSummaries", []):
    marker = "  <-- selected" if kb["knowledgeBaseId"] == KNOWLEDGE_BASE_ID else ""
    print(f"{kb['knowledgeBaseId']}  {kb['status']:<10} {kb['name']}{marker}")

In [ ]:
# Sanity check: can we actually query the knowledge base?
kb_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

probe = kb_runtime.retrieve(
    knowledgeBaseId=KNOWLEDGE_BASE_ID,
    retrievalQuery={"text": "children's menu"},
    retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 2}},
)
results = probe.get("retrievalResults", [])
print(f"{len(results)} passage(s) returned\n")
for r in results:
    print(r["content"]["text"][:300].replace("\n", " "), "...\n")

## Step 2 - The bookings table

One DynamoDB table keyed on `booking_id`. The agent's tools read and write it directly.

In [ ]:
dynamodb = boto3.client("dynamodb", region_name=REGION)


def create_bookings_table(table_name):
    """Create the bookings table if it does not already exist."""
    try:
        dynamodb.create_table(
            TableName=table_name,
            KeySchema=[{"AttributeName": "booking_id", "KeyType": "HASH"}],
            AttributeDefinitions=[{"AttributeName": "booking_id", "AttributeType": "S"}],
            BillingMode="PAY_PER_REQUEST",
        )
        print(f"Creating table {table_name} ...")
        dynamodb.get_waiter("table_exists").wait(TableName=table_name)
        print("Table is ACTIVE.")
    except dynamodb.exceptions.ResourceInUseException:
        print(f"Table {table_name} already exists - reusing it.")


create_bookings_table(TABLE_NAME)
TABLE_ARN = dynamodb.describe_table(TableName=TABLE_NAME)["Table"]["TableArn"]
print(TABLE_ARN)

## Step 3 - The agent

This is the whole agent. Read it before running the cell - it is the part worth
understanding, and it is short.

Five tools: `search_menu`, `create_booking`, `get_booking_details`, `list_bookings`
and `delete_booking`. `list_bookings` is what lets a customer ask "show me my bookings"
without already knowing a booking id.

Three things to notice:

1. **Tools are just functions.** The `@tool` decorator plus the type hints and the
   docstring give the model everything it needs to know about a tool. There is no
   separate JSON schema to keep in sync.
2. **The knowledge base is a tool too.** `search_menu` calls `bedrock-agent-runtime.retrieve`
   against the KB you configured above. Because it is an ordinary tool, the model decides
   when to use it, and you can see exactly what it returns.
3. **Memory is explicit.** `load_history` reads the last turns of this session before
   answering; `save_turn` writes the new turn afterwards. That is what lets the agent
   understand a follow-up question like "which of those are vegetarian?".

`BedrockAgentCoreApp` + `@app.entrypoint` is the AgentCore Runtime contract: it turns
`invoke()` into a `POST /invocations` HTTP endpoint with a `/ping` health check, which is
what the managed runtime calls.

In [ ]:
%%writefile restaurant_agent.py
"""Restaurant booking agent for Amazon Bedrock AgentCore Runtime.

This single file is the whole agent:

  * four tools - three that read and write bookings in DynamoDB, one that
    searches the menu knowledge base
  * short-term memory, so the agent follows a multi-turn conversation
  * an entrypoint that AgentCore Runtime serves as an HTTP endpoint

Everything below runs inside a container that AgentCore starts for you.
"""

import os
from datetime import datetime, timezone
from typing import Any, Dict, List
from uuid import uuid4

import boto3
from boto3.dynamodb.conditions import Attr
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# ---------------------------------------------------------------------------
# Configuration - every value is injected as an environment variable at launch
# time, so there are no hardcoded resource IDs in this file.
# ---------------------------------------------------------------------------
REGION = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
TABLE_NAME = os.environ.get("BOOKINGS_TABLE", "restaurant_bookings")
KNOWLEDGE_BASE_ID = os.environ.get("KNOWLEDGE_BASE_ID", "")
MODEL_ID = os.environ.get("MODEL_ID", "us.amazon.nova-2-lite-v1:0")

# AgentCore Runtime injects this automatically when the agent is configured
# with memory_mode="STM_ONLY". There is no memory id to copy around by hand.
MEMORY_ID = os.environ.get("BEDROCK_AGENTCORE_MEMORY_ID")

# How many previous turns of the conversation to replay into the model.
MEMORY_TURNS = 10

bookings_table = boto3.resource("dynamodb", region_name=REGION).Table(TABLE_NAME)
kb_runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)
memory_client = MemoryClient(region_name=REGION) if MEMORY_ID else None

SYSTEM_PROMPT = """You are a restaurant agent. You help clients look up, create and cancel
table bookings, and answer questions about the restaurant's menu.

Rules:
- Use search_menu for any question about dishes, ingredients, prices or specials.
  Answer only from what that tool returns; never invent menu items.
- Before calling create_booking you need a date (YYYY-MM-DD), a time (HH:MM, 24h),
  a name for the reservation, and the number of guests. Ask for whatever is missing.
- After creating a booking, always tell the customer the booking id.
- When a customer asks about "my bookings" and you do not have a booking id, call
  list_bookings with their name. Only ask for their name if you do not know it yet -
  never ask for a booking id you have not given them.
- Be concise and friendly.
- Reply in short Markdown: bold for key values, bullet lists for several items. Do not
  use headings, and do not use a heading for a one-line answer."""


# ---------------------------------------------------------------------------
# Tools. The function signature and the docstring ARE the schema - Strands reads
# them and tells the model what each tool is for and what it accepts. There is
# no separate JSON schema to keep in sync.
# ---------------------------------------------------------------------------
@tool
def get_booking_details(booking_id: str) -> dict:
    """Retrieve the details of an existing restaurant booking.

    Args:
        booking_id: The ID of the booking to retrieve.
    """
    response = bookings_table.get_item(Key={"booking_id": booking_id})
    item = response.get("Item")
    if item is None:
        return {"message": f"No booking found with ID {booking_id}"}
    return item


@tool
def list_bookings(name: str) -> list:
    """List every booking held under a customer's name.

    Use this when a customer asks about "my bookings" and has not given a
    booking id.

    Args:
        name: The name the reservations were made under.
    """
    items = []
    scan_kwargs = {"FilterExpression": Attr("name").eq(name)}

    while True:
        response = bookings_table.scan(**scan_kwargs)
        items.extend(response.get("Items", []))
        last_key = response.get("LastEvaluatedKey")
        if not last_key:
            break
        scan_kwargs["ExclusiveStartKey"] = last_key

    return items


@tool
def create_booking(date: str, name: str, hour: str, num_guests: int) -> dict:
    """Create a new restaurant booking.

    Args:
        date: The date of the booking in the format YYYY-MM-DD.
        name: Name to identify the reservation.
        hour: The hour of the booking in the format HH:MM.
        num_guests: The number of guests for the booking.
    """
    booking_id = str(uuid4())[:8]
    bookings_table.put_item(
        Item={
            "booking_id": booking_id,
            "date": date,
            "name": name,
            "hour": hour,
            "num_guests": int(num_guests),
        }
    )
    return {"booking_id": booking_id}


@tool
def delete_booking(booking_id: str) -> dict:
    """Delete an existing restaurant booking.

    Args:
        booking_id: The ID of the booking to delete.
    """
    bookings_table.delete_item(Key={"booking_id": booking_id})
    return {"message": f"Booking with ID {booking_id} deleted successfully"}


@tool
def search_menu(query: str) -> str:
    """Search the restaurant's menus and weekly specials.

    Use this for any question about dishes, ingredients, prices, the children's
    menu, the dinner menu or the specials of the week.

    Args:
        query: What the customer wants to know about the menu.
    """
    if not KNOWLEDGE_BASE_ID:
        return "The menu knowledge base is not configured."

    response = kb_runtime.retrieve(
        knowledgeBaseId=KNOWLEDGE_BASE_ID,
        retrievalQuery={"text": query},
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5}},
    )
    passages = [r["content"]["text"] for r in response.get("retrievalResults", [])]
    if not passages:
        return "Nothing in the menu matches that question."
    return "\n\n---\n\n".join(passages)


TOOLS = [get_booking_details, list_bookings, create_booking, delete_booking, search_menu]


# ---------------------------------------------------------------------------
# AgentCore Memory. Read the last few turns of this session before answering,
# write the new turn afterwards. That is what lets the agent understand
# "which of those are vegetarian?" as a follow-up question.
# ---------------------------------------------------------------------------
def load_history(actor_id: str, session_id: str) -> List[Dict[str, Any]]:
    """Return previous turns of this session in the format Strands expects."""
    if memory_client is None:
        return []

    turns = memory_client.get_last_k_turns(
        memory_id=MEMORY_ID,
        actor_id=actor_id,
        session_id=session_id,
        k=MEMORY_TURNS,
    )

    messages: List[Dict[str, Any]] = []
    for turn in reversed(turns):  # the API returns the newest turn first
        for message in turn:
            role = "user" if message.get("role", "").upper() == "USER" else "assistant"
            text = message.get("content", {}).get("text", "")
            if text:
                messages.append({"role": role, "content": [{"text": text}]})
    return messages


def save_turn(actor_id: str, session_id: str, user_text: str, agent_text: str) -> None:
    """Persist this turn so the next invocation can read it back."""
    if memory_client is None:
        return
    memory_client.create_event(
        memory_id=MEMORY_ID,
        actor_id=actor_id,
        session_id=session_id,
        messages=[(user_text, "USER"), (agent_text, "ASSISTANT")],
    )


# ---------------------------------------------------------------------------
# Entrypoint. AgentCore Runtime turns this into POST /invocations for you,
# with a /ping health check alongside it.
# ---------------------------------------------------------------------------
app = BedrockAgentCoreApp()


def build_system_prompt(customer_name: str, today: str) -> str:
    """Fold per-request context into the system prompt."""
    extras = [f"Today's date is {today}."]
    if customer_name:
        extras.append(f"The customer you are talking to is called {customer_name}.")
        extras.append("Use that name for the reservation unless they give another one.")
    return SYSTEM_PROMPT + "\n\n" + " ".join(extras)


@app.entrypoint
def invoke(payload, context):
    """Handle one turn of the conversation.

    Payload fields:
        prompt        (required) what the customer said
        customer_name (optional) name to use for reservations
        today         (optional) the date the agent should treat as today
        actor_id      (optional) who is talking - memory is scoped per actor
    """
    user_text = payload.get("prompt")
    if not isinstance(user_text, str) or not user_text.strip():
        return {"error": "'prompt' must be a non-empty string"}

    customer_name = payload.get("customer_name", "")
    today = payload.get("today") or datetime.now(timezone.utc).strftime("%Y-%m-%d")
    actor_id = payload.get("actor_id", "default-customer")
    session_id = getattr(context, "session_id", None) or "local-session"

    agent = Agent(
        model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
        system_prompt=build_system_prompt(customer_name, today),
        tools=TOOLS,
        messages=load_history(actor_id, session_id),
    )

    result = agent(user_text)
    try:
        agent_text = result.message["content"][0]["text"]
    except (AttributeError, KeyError, IndexError, TypeError):
        agent_text = str(result)

    save_turn(actor_id, session_id, user_text, agent_text)
    return {"result": agent_text, "session_id": session_id}


if __name__ == "__main__":
    app.run()

And the dependencies that get installed **inside** the deployed container. Keep this list
minimal - it is baked into the image.

In [ ]:
%%writefile requirements.txt
# Dependencies installed INSIDE the AgentCore Runtime container.
# Keep this list small - it is baked into the deployed image.
bedrock-agentcore
strands-agents
boto3

## Step 4 - Turn on Observability (one time per AWS account)

Agents hosted on AgentCore Runtime are instrumented automatically, but CloudWatch needs
**Transaction Search** enabled once per account before the spans become visible.

Three independent settings, so the cell below runs them separately - any one of them may
already be in place from an earlier project, and that is fine:

1. A CloudWatch Logs resource policy letting X-Ray write spans
2. Trace segments destined for CloudWatch Logs
3. An indexing rule saying **what percentage of spans to index**

Step 3 matters more than it looks. The default is **1%**, which means a handful of test
invocations will index *nothing* and `obs.list()` will report "No spans found". The cell
sets it to 100% so everything you do in this notebook shows up. Lower it for production.

If the cell fails on permissions, do the same thing in the console:
**CloudWatch -> Settings -> Account -> X-Ray traces -> Transaction Search -> Enable**.

In [ ]:
import json

logs = boto3.client("logs", region_name=REGION)
xray = boto3.client("xray", region_name=REGION)

resource_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "TransactionSearchXRayAccess",
        "Effect": "Allow",
        "Principal": {"Service": "xray.amazonaws.com"},
        "Action": "logs:PutLogEvents",
        "Resource": [
            f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
            f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*",
        ],
        "Condition": {
            "ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
            "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
        },
    }],
}

# 1. Let X-Ray write spans into CloudWatch Logs.
try:
    logs.put_resource_policy(
        policyName="TransactionSearchAccess",
        policyDocument=json.dumps(resource_policy),
    )
    print("1/3 resource policy set")
except Exception as e:
    print(f"1/3 resource policy: {e}")

# 2. Send trace segments to CloudWatch Logs.
#    "The destination is already set to CloudWatchLogs" just means it was already on.
try:
    xray.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("2/3 trace destination set to CloudWatchLogs")
except Exception as e:
    print(f"2/3 trace destination: {e}")

# 3. Index 100% of spans so this notebook's traces are all visible.
try:
    xray.update_indexing_rule(
        Name="Default",
        Rule={"Probabilistic": {"DesiredSamplingPercentage": 100}},
    )
    print("3/3 indexing 100% of spans")
except Exception as e:
    print(f"3/3 indexing rule: {e}")

In [ ]:
# Confirm what is actually in effect. Both lines should look right before you rely on traces.
print(xray.get_trace_segment_destination())
for rule in xray.get_indexing_rules().get("IndexingRules", []):
    print(rule)

## Step 5 - Configure the runtime

`configure()` writes a local `.bedrock_agentcore.yaml` describing how to build and deploy
the agent. Nothing is created in AWS yet.

The two arguments that do the most work for you:

- `auto_create_execution_role=True` - builds the IAM role the agent runs as, already
  scoped to invoke Bedrock models, write CloudWatch logs, emit traces, and use its own
  memory store.
- `memory_mode="STM_ONLY"` - provisions an **AgentCore Memory** store for short-term
  (within-session) conversation memory, and injects its ID into the container as
  `BEDROCK_AGENTCORE_MEMORY_ID`. That is the variable `restaurant_agent.py` reads.

Use `"STM_AND_LTM"` instead if you also want long-term memory that persists across
sessions and extracts facts and preferences over time.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()

configure_result = runtime.configure(
    entrypoint=ENTRYPOINT,
    agent_name=AGENT_NAME,
    requirements_file="requirements.txt",
    region=REGION,
    auto_create_execution_role=True,   # IAM role for the agent
    auto_create_ecr=True,              # ECR repo for the container image
    memory_mode="STM_ONLY",            # AgentCore Memory: short-term conversation memory
    non_interactive=True,
)

print(f"Config file: {configure_result.config_path}")
print(f"Memory ID:   {configure_result.memory_id}")

## Step 6 - Deploy to AgentCore Runtime

![Build and deploy](./images/deployment-architecture.png)

This is what the single `launch()` call below actually does. Watch the output and you will
see each of these steps go past:

1. Zips this folder and uploads it to an **S3** source bucket
   (`bedrock-agentcore-codebuild-sources-{account}-{region}`)
2. Creates an **AWS CodeBuild** project that builds the container on ARM64 hardware
   (`ARM_CONTAINER`, `amazonlinux2-aarch64-standard:3.0`) - this is why you do not need
   Docker locally
3. CodeBuild pushes the finished image to **Amazon ECR**
4. Creates the **AgentCore Runtime** and its `DEFAULT` endpoint, pointing at that image

Along the way it also creates the IAM execution role, the AgentCore Memory store, the
CodeBuild service role, and the CloudWatch log groups.

There is no CodePipeline here - CodeBuild is invoked directly. Re-running `launch()` after
you edit the agent rebuilds the image and updates the runtime in place.

**The first run takes 3-6 minutes.** Later ones are faster because the dependency layers
are cached.

The environment variables passed here are how the agent learns about your table, your
knowledge base and your model - the agent file itself has no hardcoded IDs.

In [ ]:
launch_result = runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BOOKINGS_TABLE": TABLE_NAME,
        "KNOWLEDGE_BASE_ID": KNOWLEDGE_BASE_ID,
        "MODEL_ID": MODEL_ID,
    },
)

print(f"Mode:      {launch_result.mode}")
print(f"Agent ID:  {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")

In [ ]:
import time

# Wait until the endpoint is READY before invoking.
for _ in range(60):
    status = runtime.status()
    endpoint = status.endpoint or {}
    if "error" in endpoint:
        print(f"Could not read endpoint status: {endpoint['error']}")
        break
    endpoint_status = endpoint.get("status", "UNKNOWN")
    print(f"endpoint: {endpoint_status}")
    if endpoint_status in ("READY", "CREATE_FAILED", "UPDATE_FAILED"):
        break
    time.sleep(15)

EXECUTION_ROLE_ARN = status.config.execution_role
MEMORY_ID = status.config.memory_id
print(f"\nExecution role: {EXECUTION_ROLE_ARN}")
print(f"Memory ID:      {MEMORY_ID}")
print(f"Memory status:  {status.config.memory_status}")

## Step 7 - Give the agent access to your data

The auto-created execution role already covers Bedrock model invocation, logs, traces and
AgentCore Memory. It does **not** know about your DynamoDB table or your Knowledge Base -
those are yours, so you grant them explicitly with one inline policy.

In [ ]:
iam = boto3.client("iam")
role_name = EXECUTION_ROLE_ARN.split("/")[-1]

agent_data_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BookingsTableAccess",
            "Effect": "Allow",
            "Action": [
                "dynamodb:GetItem",
                "dynamodb:PutItem",
                "dynamodb:DeleteItem",
                "dynamodb:Scan",       # list_bookings scans by customer name
            ],
            "Resource": TABLE_ARN,
        },
        {
            "Sid": "MenuKnowledgeBaseAccess",
            "Effect": "Allow",
            "Action": ["bedrock:Retrieve"],
            "Resource": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/{KNOWLEDGE_BASE_ID}",
        },
    ],
}

iam.put_role_policy(
    RoleName=role_name,
    PolicyName="RestaurantAgentDataAccess",
    PolicyDocument=json.dumps(agent_data_policy),
)
print(f"Policy attached to {role_name}")

time.sleep(10)  # let IAM propagate before the first invocation

## Step 8 - Talk to the agent

`runtime.invoke()` sends a payload to the deployed endpoint. Reuse a `session_id` to
continue a conversation, generate a new one to start fresh. It has to be at least
33 characters.

In [ ]:
import uuid


def new_session():
    """AgentCore requires a session id of at least 33 characters."""
    return f"{uuid.uuid4()}-{uuid.uuid4().hex[:8]}"


def decode(event):
    """The runtime hands our dict back as a JSON string - turn it into a dict."""
    if isinstance(event, (bytes, bytearray)):
        event = event.decode("utf-8")
    if isinstance(event, str):
        try:
            event = json.loads(event)
        except json.JSONDecodeError:
            return event
    return event


def ask(question, session_id, **extras):
    """Send one turn to the agent and print the reply."""
    payload = {"prompt": question, **extras}
    response = runtime.invoke(payload, session_id=session_id)

    reply = None
    for event in response.get("response", []):
        event = decode(event)
        if isinstance(event, dict):
            if "result" in event:
                reply = event["result"]
            elif "error" in event:
                reply = f"ERROR: {event['error']}"
            else:
                reply = event
        else:
            reply = event
        print(reply)
    return reply

### Asking the knowledge base

The model decides to call `search_menu`, which retrieves from your Knowledge Base.

In [ ]:
session_id = new_session()

ask("What is in the childrens menu?", session_id)

### A follow-up, in the same session

This is AgentCore Memory doing its job. The agent has no idea what "those options" means
on its own - `load_history` replayed the previous turn into the model.

In [ ]:
ask("Which of those options are vegetarian?", session_id)

### Using a tool: creating a booking

Now the agent calls `create_booking`, which writes straight to DynamoDB.

In [ ]:
ask(
    "Hi, I am Maria. I want to create a booking for 4 people, at 21:00 on 2026-05-05.",
    session_id,
)

In [ ]:
import pandas as pd

bookings = boto3.resource("dynamodb", region_name=REGION).Table(TABLE_NAME)


def show_bookings():
    """Scan the table so we can verify what the agent actually did."""
    items = bookings.scan().get("Items", [])
    return pd.DataFrame(items) if items else pd.DataFrame(columns=["booking_id"])


show_bookings()

### Passing context into the prompt

`customer_name` and `today` are payload fields that `build_system_prompt()` folds into the
system prompt, so the agent knows who it is talking to without the customer typing it.

New session id, so the agent has no memory of Maria.

In [ ]:
session_id_2 = new_session()

ask(
    "I want to create a booking for 2 people, at 20:00 on 2026-05-06.",
    session_id_2,
    customer_name="John",
    today="2026-05-01",
)

In [ ]:
show_bookings()

### Reading and deleting a booking

The agent remembers the booking id it just created, so it can look it up and delete it
without being told the id again.

In [ ]:
ask("Get the details for the booking you just created.", session_id_2)

In [ ]:
ask("I want to delete that booking.", session_id_2)

In [ ]:
show_bookings()

## Step 9 - Observability

Every invocation above produced an OpenTelemetry trace: the model calls, the tool calls,
their inputs and outputs, and the latency of each. You did not add a single line of
instrumentation.

`obs.list()` shows the traces in a session; `obs.show()` expands one of them.

**If these come back "No spans found":**

- Give it 5-10 minutes. Spans are not queryable the instant they are emitted.
- Re-check Step 4. If the indexing percentage is still at the default 1%, a handful of
  invocations will index nothing at all.
- Run a couple more `ask()` calls after fixing the sampling, then look again.

In [ ]:
from bedrock_agentcore_starter_toolkit import Observability

obs = Observability(agent_id=launch_result.agent_id, region=REGION)

obs.list(session_id=session_id_2)

In [ ]:
obs.show(session_id=session_id_2)

You can also open the same data in the console:
**CloudWatch -> GenAI Observability -> Bedrock AgentCore**.

Raw logs, if you want them:

```
aws logs tail /aws/bedrock-agentcore/runtimes/<agent_id>-DEFAULT --follow
```

## Step 10 - The end-to-end picture

You can talk to the agent from this notebook. Here is what it looks like once a real
application is in front of it.

![End to end architecture](./images/end-to-end-architecture.png)

Reading it left to right:

| Hop | What happens |
| --- | --- |
| **Customer browser** | Loads the chat page and posts what the customer typed |
| **Web server** (`ui/app.py`) | Flask serves the page and forwards the turn server-side, so the API URL never reaches the browser and you need no CORS |
| **Amazon API Gateway** | HTTP API route, proxies the request to Lambda |
| **AWS Lambda** (`lambda_function.py`) | Calls `bedrock-agentcore:InvokeAgentRuntime` with the prompt and a session id - this is the `ask()` helper above, deployed |
| **AgentCore Runtime** | Runs the agent: picks a tool, calls the model, reads and writes Memory, emits traces |
| **DynamoDB / Knowledge Base** | Where the booking tools and `search_menu` actually land |

One endpoint covers all four capabilities. The agent chooses the tool from what the
customer says - "book a table for 4" reaches `create_booking`, "what's on the children's
menu" reaches `search_menu`.

To wire it up, copy the **Agent Runtime ARN** printed below into the `AGENT_RUNTIME_ARN`
environment variable on your Lambda. Full setup - IAM policy, timeout, running the Flask
app - is in `README.md`.

In [ ]:
print("Set this as the AGENT_RUNTIME_ARN environment variable on your Lambda:\n")
print(launch_result.agent_arn)

## Step 11 - Clean up

Run this when you are done, so the runtime, ECR images, memory store and table do not
keep costing money.

In [ ]:
# Preview first - see what would be removed without removing it.
runtime.destroy(dry_run=True)

In [ ]:
# Removes the AgentCore Runtime and endpoint, the memory store, the ECR images and
# repository, the CodeBuild role, and the execution role (if no other agent uses it).
runtime.destroy(delete_ecr_repo=True)

In [ ]:
# destroy() usually deletes the execution role too (and our inline policy with it).
# This is only here in case the role is shared with another agent and was kept.
try:
    iam.delete_role_policy(RoleName=role_name, PolicyName="RestaurantAgentDataAccess")
    print("Inline policy deleted.")
except iam.exceptions.NoSuchEntityException:
    print("Role or inline policy already removed by destroy().")

# Drop the bookings table.
dynamodb.delete_table(TableName=TABLE_NAME)
print(f"Deleting table {TABLE_NAME} ...")
dynamodb.get_waiter("table_not_exists").wait(TableName=TABLE_NAME)
print("Done.")

> The Knowledge Base is **not** deleted - it was yours before this notebook and it stays.

## Where to go next

- Swap `memory_mode="STM_ONLY"` for `"STM_AND_LTM"` and ask the agent to remember a
  customer's dietary preference across two different sessions.
- Add a tool - `list_bookings_for_date`, say - and redeploy with `runtime.launch()`.
- Put the booking tools behind an **AgentCore Gateway** so other agents can call them
  over MCP.
- Add **AgentCore Identity** so each caller's own identity, not a shared role, decides
  which bookings they can see.